### Шаг 1: Установка системных зависимостей OpenMPI и библиотеки `mpi4py`

In [3]:
!apt-get update
!apt-get install -y openmpi-bin libopenmpi-dev
import sys
!{sys.executable} -m pip install --no-cache-dir mpi4py

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [81.1 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,088 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,149 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.6 MB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60

### Шаг 2: Проверка GPU и доступности `numba.cuda`

In [ ]:
#1.1 Domain Decomposition & Mathematical Formulation

In [9]:
import sys
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# ----------------------------------------------------------------- constants
N = 512                 # global interior cells per direction
PY, PX = 2, 2           # process grid
LAMBDA = 0.15
TILE = 16               # thread block = TILE x TILE
DT_SAFETY = 0.8


# ------------------------------------------------------------ physical fields
def build_fields(gy0, gx0, h, w, dx):
    """kappa and S on a padded block. Local index a <-> padded-global index gy0+a;
    padded-global index 0 / N+1 are the physical boundaries (y,x = 0 / 1)."""
    y = (gy0 + np.arange(h)) * dx
    x = (gx0 + np.arange(w)) * dx
    Y, X = np.meshgrid(y, x, indexing="ij")
    kappa = 1.0 + 0.5 * np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)   # in [0.5, 1.5]
    S = 50.0 * np.exp(-((X - 0.5) ** 2 + (Y - 0.5) ** 2) / (2 * 0.05 ** 2))
    return np.ascontiguousarray(kappa), np.ascontiguousarray(S)


# --------------------------------------------------------------------- kernels
@cuda.jit
def pack_rows_kernel(u, send_n, send_s, ny, nx):
    j = cuda.grid(1)
    if j < nx:
        send_n[j] = u[1, j + 1]        # first owned row  -> north neighbour
        send_s[j] = u[ny, j + 1]       # last owned row   -> south neighbour


@cuda.jit
def pack_cols_kernel(u, send_w, send_e, ny, nx):
    i = cuda.grid(1)
    if i < ny:
        send_w[i] = u[i + 1, 1]        # first owned col  -> west neighbour
        send_e[i] = u[i + 1, nx]       # last owned col   -> east neighbour


@cuda.jit
def unpack_rows_kernel(u, recv_n, recv_s, ny, nx):
    j = cuda.grid(1)
    if j < nx:
        u[0, j + 1] = recv_n[j]        # north ghost row
        u[ny + 1, j + 1] = recv_s[j]   # south ghost row


@cuda.jit
def unpack_cols_kernel(u, recv_w, recv_e, ny, nx):
    i = cuda.grid(1)
    if i < ny:
        u[i + 1, 0] = recv_w[i]        # west ghost col
        u[i + 1, nx + 1] = recv_e[i]   # east ghost col


@cuda.jit
def stencil_kernel(u, un, kap, src, i0, i1, j0, j1, dt, inv_dx2, inv_dy2, lam):
    """Updates cells [i0,i1) x [j0,j1) of the padded array using a shared-memory tile.
    Only cells actually needed are loaded, so a region that excludes the outer ring
    never touches ghost cells (this makes the overlap race-free)."""
    s = cuda.shared.array((TILE + 2, TILE + 2), float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    j = j0 + cuda.blockIdx.x * TILE + tx
    i = i0 + cuda.blockIdx.y * TILE + ty
    valid = (i < i1) and (j < j1)

    if valid:
        s[ty + 1, tx + 1] = u[i, j]
        if ty == 0:
            s[0, tx + 1] = u[i - 1, j]
        if ty == TILE - 1 or i == i1 - 1:
            s[ty + 2, tx + 1] = u[i + 1, j]
        if tx == 0:
            s[ty + 1, 0] = u[i, j - 1]
        if tx == TILE - 1 or j == j1 - 1:
            s[ty + 1, tx + 2] = u[i, j + 1]
    cuda.syncthreads()

    if valid:
        c = s[ty + 1, tx + 1]
        uN = s[ty, tx + 1]
        uS = s[ty + 2, tx + 1]
        uW = s[ty + 1, tx]
        uE = s[ty + 1, tx + 2]
        k = kap[i, j]
        kN = 0.5 * (kap[i - 1, j] + k)
        kS = 0.5 * (kap[i + 1, j] + k)
        kW = 0.5 * (kap[i, j - 1] + k)
        kE = 0.5 * (kap[i, j + 1] + k)
        diff_x = (kE * (uE - c) - kW * (c - uW)) * inv_dx2
        diff_y = (kS * (uS - c) - kN * (c - uN)) * inv_dy2
        un[i, j] = c + dt * (diff_x + diff_y - lam * c * c * c + src[i, j])


# ------------------------------------------------------------------ main solver
def run(nsteps, verify):
    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    if size != PY * PX:
        if rank == 0:
            print(f"Need exactly {PY * PX} ranks, got {size}")
        sys.exit(1)

    # ---- Cartesian topology
    cart = comm.Create_cart(dims=[PY, PX], periods=[False, False], reorder=False)
    cy, cx = cart.Get_coords(rank)
    north, south = cart.Shift(0, 1)    # (source = row-1, dest = row+1)
    west, east = cart.Shift(1, 1)      # (source = col-1, dest = col+1)
    NULL = MPI.PROC_NULL

    ny, nx = N // PY, N // PX          # 256 x 256 owned
    dx = dy = 1.0 / (N + 1)
    dt = DT_SAFETY * dx * dx / (4.0 * 1.5)
    inv_dx2, inv_dy2 = 1.0 / dx ** 2, 1.0 / dy ** 2

    # ---- GPU setup (all ranks share the single T4)
    cuda.select_device(rank % len(cuda.gpus))
    s_comp = cuda.stream()
    s_comm = cuda.stream()

    kap_h, src_h = build_fields(cy * ny, cx * nx, ny + 2, nx + 2, dx)
    d_kap = cuda.to_device(kap_h)
    d_src = cuda.to_device(src_h)
    zeros_pad = np.zeros((ny + 2, nx + 2), dtype=np.float64)
    d_u = cuda.to_device(zeros_pad)
    d_un = cuda.to_device(zeros_pad)   # ghosts at physical boundaries stay 0 (Dirichlet)

    # ---- device halo buffers + pinned host staging buffers
    def dbuf(n):
        return cuda.device_array(n, dtype=np.float64)

    def hbuf(n):
        a = cuda.pinned_array(n, dtype=np.float64)
        a[:] = 0.0
        return a

    d_send = {"n": dbuf(nx), "s": dbuf(nx), "w": dbuf(ny), "e": dbuf(ny)}
    d_recv = {"n": dbuf(nx), "s": dbuf(nx), "w": dbuf(ny), "e": dbuf(ny)}
    h_send = {"n": hbuf(nx), "s": hbuf(nx), "w": hbuf(ny), "e": hbuf(ny)}
    h_recv = {"n": hbuf(nx), "s": hbuf(nx), "w": hbuf(ny), "e": hbuf(ny)}
    neigh = {"n": north, "s": south, "w": west, "e": east}

    # ---- launch configs
    tpb1d = 128
    g_rows = (nx + tpb1d - 1) // tpb1d
    g_cols = (ny + tpb1d - 1) // tpb1d
    block2d = (TILE, TILE)

    def launch_stencil(stream, u, un, i0, i1, j0, j1):
        if i1 <= i0 or j1 <= j0:
            return
        grid = ((j1 - j0 + TILE - 1) // TILE, (i1 - i0 + TILE - 1) // TILE)
        stencil_kernel[grid, block2d, stream](u, un, d_kap, d_src, i0, i1, j0, j1,
                                              dt, inv_dx2, inv_dy2, LAMBDA)

    def step(u, un):
        # (1) comm stream: pack boundary cells, copy device -> pinned host
        pack_rows_kernel[g_rows, tpb1d, s_comm](u, d_send["n"], d_send["s"], ny, nx)
        pack_cols_kernel[g_cols, tpb1d, s_comm](u, d_send["w"], d_send["e"], ny, nx)
        for k in "nsw" + "e":
            if neigh[k] != NULL:
                d_send[k].copy_to_host(h_send[k], stream=s_comm)

        # (2) compute stream: interior cells (do not depend on ghosts) -- overlaps with comm
        launch_stencil(s_comp, u, un, 2, ny, 2, nx)

        # (3) MPI halo exchange on pinned buffers
        s_comm.synchronize()
        reqs = [
            cart.Isend(h_send["n"], dest=north, tag=0),   # moving north
            cart.Irecv(h_recv["s"], source=south, tag=0),
            cart.Isend(h_send["s"], dest=south, tag=1),   # moving south
            cart.Irecv(h_recv["n"], source=north, tag=1),
            cart.Isend(h_send["w"], dest=west, tag=2),    # moving west
            cart.Irecv(h_recv["e"], source=east, tag=2),
            cart.Isend(h_send["e"], dest=east, tag=3),    # moving east
            cart.Irecv(h_recv["w"], source=west, tag=3),
        ]
        MPI.Request.Waitall(reqs)

        # (4) comm stream: host -> device, unpack into ghost cells, then boundary ring
        for k in "nsw" + "e":
            if neigh[k] != NULL:
                d_recv[k].copy_to_device(h_recv[k], stream=s_comm)
        unpack_rows_kernel[g_rows, tpb1d, s_comm](u, d_recv["n"], d_recv["s"], ny, nx)
        unpack_cols_kernel[g_cols, tpb1d, s_comm](u, d_recv["w"], d_recv["e"], ny, nx)
        launch_stencil(s_comm, u, un, 1, 2, 1, nx + 1)             # top row
        launch_stencil(s_comm, u, un, ny, ny + 1, 1, nx + 1)       # bottom row
        launch_stencil(s_comm, u, un, 2, ny, 1, 2)                 # left column
        launch_stencil(s_comm, u, un, 2, ny, nx, nx + 1)           # right column

        s_comm.synchronize()
        s_comp.synchronize()

    # ---- warm-up (JIT compile), then reset state
    step(d_u, d_un)
    d_u.copy_to_device(zeros_pad)
    d_un.copy_to_device(zeros_pad)
    cuda.synchronize()
    comm.Barrier()

    # ---- time loop
    t0 = MPI.Wtime()
    for _ in range(nsteps):
        step(d_u, d_un)
        d_u, d_un = d_un, d_u
    cuda.synchronize()
    comm.Barrier()
    t1 = MPI.Wtime()

    # ---- diagnostics
    local = d_u.copy_to_host()[1:ny + 1, 1:nx + 1].copy()
    l2 = comm.allreduce(float(np.sum(local ** 2)), op=MPI.SUM) ** 0.5
    umax = comm.allreduce(float(local.max()), op=MPI.MAX)
    if rank == 0:
        print(f"steps={nsteps}  dt={dt:.3e}  time={t1 - t0:.3f}s  "
              f"({(t1 - t0) / nsteps * 1e3:.3f} ms/step)")
        print(f"||u||_2 = {l2:.6e}   max(u) = {umax:.6e}")

    # ---- optional validation against a serial NumPy reference
    if verify:
        recv = np.empty((size, ny, nx), dtype=np.float64) if rank == 0 else None
        comm.Gather(local, recv, root=0)
        if rank == 0:
            full = np.empty((N, N))
            for r in range(size):
                ry, rx = cart.Get_coords(r)
                full[ry * ny:(ry + 1) * ny, rx * nx:(rx + 1) * nx] = recv[r]
            ref = serial_reference(nsteps, dt, dx)
            print(f"max |u_mpi_cuda - u_serial| = {np.abs(full - ref).max():.3e}")


def serial_reference(nsteps, dt, dx):
    kap, src = build_fields(0, 0, N + 2, N + 2, dx)
    u = np.zeros((N + 2, N + 2))
    inv = 1.0 / dx ** 2
    kc = kap[1:-1, 1:-1]
    kN = 0.5 * (kap[:-2, 1:-1] + kc)
    kS = 0.5 * (kap[2:, 1:-1] + kc)
    kW = 0.5 * (kap[1:-1, :-2] + kc)
    kE = 0.5 * (kap[1:-1, 2:] + kc)
    S = src[1:-1, 1:-1]
    for _ in range(nsteps):
        c = u[1:-1, 1:-1]
        rhs = ((kE * (u[1:-1, 2:] - c) - kW * (c - u[1:-1, :-2])) * inv
               + (kS * (u[2:, 1:-1] - c) - kN * (c - u[:-2, 1:-1])) * inv
               - LAMBDA * c ** 3 + S)
        u[1:-1, 1:-1] = c + dt * rhs
    return u[1:-1, 1:-1].copy()


if __name__ == "__main__":
    # Игнорируем технические аргументы Jupyter ядра (например, '-f')
    args = [a for a in sys.argv[1:] if not a.startswith("--") and not a.startswith("-f") and ".json" not in a]
    nsteps = int(args[0]) if args else 500
    run(nsteps, verify="--verify" in sys.argv)

Need exactly 4 ranks, got 1


SystemExit: 1

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
#1.3 Dual-Stream Compute-Communication Pipeline

In [ ]:
"""
4-rank distributed 2D reaction-diffusion solver  (mpi4py + numba.cuda)

  u_t = div(kappa grad u) - lambda * u^3 + S

Features:
  * 2x2 non-periodic Cartesian MPI topology (Create_cart / Shift)
  * 256x256 owned cells per rank, padded device array 258x258 (1-cell ghost halo)
  * GPU halo pack / unpack kernels
  * page-locked (pinned) host staging buffers
  * 2D shared-memory tiled 5-point stencil kernel
  * two CUDA streams (stream_boundary / stream_interior): interior compute overlaps with halo exchange

Run (Colab):
  !mpirun --allow-run-as-root --oversubscribe -np 4 python rd_solver.py 500 --verify
Args:  [nsteps]  [--verify]
Array layout: u[iy, jx]  (row = y direction, column = x direction)
"""
import sys
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# ----------------------------------------------------------------- constants
N = 512                 # global interior cells per direction
PY, PX = 2, 2           # process grid
LAMBDA = 0.15
TILE = 16               # thread block = TILE x TILE
DT_SAFETY = 0.8


# ------------------------------------------------------------ physical fields
def build_fields(gy0, gx0, h, w, dx):
    """kappa and S on a padded block. Local index a <-> padded-global index gy0+a;
    padded-global index 0 / N+1 are the physical boundaries (y,x = 0 / 1)."""
    y = (gy0 + np.arange(h)) * dx
    x = (gx0 + np.arange(w)) * dx
    Y, X = np.meshgrid(y, x, indexing="ij")
    kappa = 1.0 + 0.5 * np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)   # in [0.5, 1.5]
    S = 50.0 * np.exp(-((X - 0.5) ** 2 + (Y - 0.5) ** 2) / (2 * 0.05 ** 2))
    return np.ascontiguousarray(kappa), np.ascontiguousarray(S)


# --------------------------------------------------------------------- kernels
@cuda.jit
def pack_halos_kernel(u, send_n, send_s, send_w, send_e, ny, nx):
    """Copy the four owned border strips (row 1, row ny, col 1, col nx)
    into contiguous 1D device send buffers."""
    t = cuda.grid(1)
    if t < nx:
        send_n[t] = u[1, t + 1]        # first owned row -> north neighbour
        send_s[t] = u[ny, t + 1]       # last owned row  -> south neighbour
    if t < ny:
        send_w[t] = u[t + 1, 1]        # first owned col -> west neighbour
        send_e[t] = u[t + 1, nx]       # last owned col  -> east neighbour


@cuda.jit
def unpack_halos_kernel(u, recv_n, recv_s, recv_w, recv_e, ny, nx):
    """Write received halos into ghost indices 0 and n+1."""
    t = cuda.grid(1)
    if t < nx:
        u[0, t + 1] = recv_n[t]
        u[ny + 1, t + 1] = recv_s[t]
    if t < ny:
        u[t + 1, 0] = recv_w[t]
        u[t + 1, nx + 1] = recv_e[t]


@cuda.jit
def tiled_stencil_kernel(u, un, kap, src, i0, i1, j0, j1, dt, inv_dx2, inv_dy2, lam):
    """Updates cells [i0,i1) x [j0,j1) of the padded array using a shared-memory tile.
    Only cells actually needed are loaded, so a region that excludes the outer ring
    never touches ghost cells (this makes the overlap race-free)."""
    s = cuda.shared.array((TILE + 2, TILE + 2), float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    j = j0 + cuda.blockIdx.x * TILE + tx
    i = i0 + cuda.blockIdx.y * TILE + ty
    valid = (i < i1) and (j < j1)

    if valid:
        s[ty + 1, tx + 1] = u[i, j]
        if ty == 0:
            s[0, tx + 1] = u[i - 1, j]
        if ty == TILE - 1 or i == i1 - 1:
            s[ty + 2, tx + 1] = u[i + 1, j]
        if tx == 0:
            s[ty + 1, 0] = u[i, j - 1]
        if tx == TILE - 1 or j == j1 - 1:
            s[ty + 1, tx + 2] = u[i, j + 1]
    cuda.syncthreads()

    if valid:
        c = s[ty + 1, tx + 1]
        uN = s[ty, tx + 1]
        uS = s[ty + 2, tx + 1]
        uW = s[ty + 1, tx]
        uE = s[ty + 1, tx + 2]
        k = kap[i, j]
        kN = 0.5 * (kap[i - 1, j] + k)
        kS = 0.5 * (kap[i + 1, j] + k)
        kW = 0.5 * (kap[i, j - 1] + k)
        kE = 0.5 * (kap[i, j + 1] + k)
        diff_x = (kE * (uE - c) - kW * (c - uW)) * inv_dx2
        diff_y = (kS * (uS - c) - kN * (c - uN)) * inv_dy2
        un[i, j] = c + dt * (diff_x + diff_y - lam * c * c * c + src[i, j])


# ------------------------------------------------------------------ main solver
def run(nsteps, verify):
    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    if size != PY * PX:
        if rank == 0:
            print(f"Need exactly {PY * PX} ranks, got {size}")
        sys.exit(1)

    # ---- Cartesian topology
    cart = comm.Create_cart(dims=[PY, PX], periods=[False, False], reorder=False)
    cy, cx = cart.Get_coords(rank)
    north, south = cart.Shift(0, 1)    # (source = row-1, dest = row+1)
    west, east = cart.Shift(1, 1)      # (source = col-1, dest = col+1)
    NULL = MPI.PROC_NULL

    ny, nx = N // PY, N // PX          # 256 x 256 owned
    dx = dy = 1.0 / (N + 1)
    dt = DT_SAFETY * dx * dx / (4.0 * 1.5)
    inv_dx2, inv_dy2 = 1.0 / dx ** 2, 1.0 / dy ** 2

    # ---- GPU setup (all ranks share the single T4)
    cuda.select_device(rank % len(cuda.gpus))
    stream_interior = cuda.stream()
    stream_boundary = cuda.stream()

    kap_h, src_h = build_fields(cy * ny, cx * nx, ny + 2, nx + 2, dx)
    d_kap = cuda.to_device(kap_h)
    d_src = cuda.to_device(src_h)
    zeros_pad = np.zeros((ny + 2, nx + 2), dtype=np.float64)
    d_u = cuda.to_device(zeros_pad)
    d_un = cuda.to_device(zeros_pad)   # ghosts at physical boundaries stay 0 (Dirichlet)

    # ---- device halo buffers + pinned host staging buffers
    def dbuf(n):
        return cuda.device_array(n, dtype=np.float64)

    def hbuf(n):
        a = cuda.pinned_array(n, dtype=np.float64)
        a[:] = 0.0
        return a

    d_send = {"n": dbuf(nx), "s": dbuf(nx), "w": dbuf(ny), "e": dbuf(ny)}
    d_recv = {"n": dbuf(nx), "s": dbuf(nx), "w": dbuf(ny), "e": dbuf(ny)}
    h_send = {"n": hbuf(nx), "s": hbuf(nx), "w": hbuf(ny), "e": hbuf(ny)}
    h_recv = {"n": hbuf(nx), "s": hbuf(nx), "w": hbuf(ny), "e": hbuf(ny)}
    neigh = {"n": north, "s": south, "w": west, "e": east}

    # ---- launch configs
    tpb1d = 128
    g_halo = (max(nx, ny) + tpb1d - 1) // tpb1d
    block2d = (TILE, TILE)

    def launch_stencil(stream, u, un, i0, i1, j0, j1):
        if i1 <= i0 or j1 <= j0:
            return
        grid = ((j1 - j0 + TILE - 1) // TILE, (i1 - i0 + TILE - 1) // TILE)
        tiled_stencil_kernel[grid, block2d, stream](u, un, d_kap, d_src, i0, i1, j0, j1,
                                              dt, inv_dx2, inv_dy2, LAMBDA)

    def step(u, un):
        # ---- Stage A (stream_boundary): pack border strips, async D2H into pinned buffers
        pack_halos_kernel[g_halo, tpb1d, stream_boundary](
            u, d_send["n"], d_send["s"], d_send["w"], d_send["e"], ny, nx)
        for k in "nswe":
            if neigh[k] != NULL:
                d_send[k].copy_to_host(h_send[k], stream=stream_boundary)
        stream_boundary.synchronize()                      # host send buffers are ready

        # ---- Stage B: post non-blocking MPI, immediately launch deep-interior stencil
        reqs = [
            cart.Irecv(h_recv["s"], source=south, tag=0),
            cart.Irecv(h_recv["n"], source=north, tag=1),
            cart.Irecv(h_recv["e"], source=east, tag=2),
            cart.Irecv(h_recv["w"], source=west, tag=3),
            cart.Isend(h_send["n"], dest=north, tag=0),   # moving north
            cart.Isend(h_send["s"], dest=south, tag=1),   # moving south
            cart.Isend(h_send["w"], dest=west, tag=2),    # moving west
            cart.Isend(h_send["e"], dest=east, tag=3),    # moving east
        ]
        launch_stencil(stream_interior, u, un, 2, ny, 2, nx)       # [2..ny-1] x [2..nx-1]

        # ---- Stage C (stream_boundary): wait, async H2D, unpack ghosts, boundary ring
        MPI.Request.Waitall(reqs)
        for k in "nswe":
            if neigh[k] != NULL:
                d_recv[k].copy_to_device(h_recv[k], stream=stream_boundary)
        unpack_halos_kernel[g_halo, tpb1d, stream_boundary](
            u, d_recv["n"], d_recv["s"], d_recv["w"], d_recv["e"], ny, nx)
        launch_stencil(stream_boundary, u, un, 1, 2, 1, nx + 1)             # row 1
        launch_stencil(stream_boundary, u, un, ny, ny + 1, 1, nx + 1)       # row ny
        launch_stencil(stream_boundary, u, un, 2, ny, 1, 2)                 # column 1
        launch_stencil(stream_boundary, u, un, 2, ny, nx, nx + 1)           # column nx

        stream_boundary.synchronize()
        stream_interior.synchronize()

    # ---- warm-up (JIT compile), then reset state
    step(d_u, d_un)
    d_u.copy_to_device(zeros_pad)
    d_un.copy_to_device(zeros_pad)
    cuda.synchronize()
    comm.Barrier()

    # ---- time loop
    t0 = MPI.Wtime()
    for _ in range(nsteps):
        step(d_u, d_un)
        d_u, d_un = d_un, d_u
    cuda.synchronize()
    comm.Barrier()
    t1 = MPI.Wtime()

    # ---- diagnostics
    local = d_u.copy_to_host()[1:ny + 1, 1:nx + 1].copy()
    l2 = comm.allreduce(float(np.sum(local ** 2)), op=MPI.SUM) ** 0.5
    umax = comm.allreduce(float(local.max()), op=MPI.MAX)
    if rank == 0:
        print(f"steps={nsteps}  dt={dt:.3e}  time={t1 - t0:.3f}s  "
              f"({(t1 - t0) / nsteps * 1e3:.3f} ms/step)")
        print(f"||u||_2 = {l2:.6e}   max(u) = {umax:.6e}")

    # ---- optional validation against a serial NumPy reference
    if verify:
        recv = np.empty((size, ny, nx), dtype=np.float64) if rank == 0 else None
        comm.Gather(local, recv, root=0)
        if rank == 0:
            full = np.empty((N, N))
            for r in range(size):
                ry, rx = cart.Get_coords(r)
                full[ry * ny:(ry + 1) * ny, rx * nx:(rx + 1) * nx] = recv[r]
            ref = serial_reference(nsteps, dt, dx)
            print(f"max |u_mpi_cuda - u_serial| = {np.abs(full - ref).max():.3e}")


def serial_reference(nsteps, dt, dx):
    kap, src = build_fields(0, 0, N + 2, N + 2, dx)
    u = np.zeros((N + 2, N + 2))
    inv = 1.0 / dx ** 2
    kc = kap[1:-1, 1:-1]
    kN = 0.5 * (kap[:-2, 1:-1] + kc)
    kS = 0.5 * (kap[2:, 1:-1] + kc)
    kW = 0.5 * (kap[1:-1, :-2] + kc)
    kE = 0.5 * (kap[1:-1, 2:] + kc)
    S = src[1:-1, 1:-1]
    for _ in range(nsteps):
        c = u[1:-1, 1:-1]
        rhs = ((kE * (u[1:-1, 2:] - c) - kW * (c - u[1:-1, :-2])) * inv
               + (kS * (u[2:, 1:-1] - c) - kN * (c - u[:-2, 1:-1])) * inv
               - LAMBDA * c ** 3 + S)
        u[1:-1, 1:-1] = c + dt * rhs
    return u[1:-1, 1:-1].copy()


if __name__ == "__main__":
    args = [a for a in sys.argv[1:] if not a.startswith("--")]
    nsteps = int(args[0]) if args else 500
    run(nsteps, verify="--verify" in sys.argv)

In [10]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python rd_solver.py 500 --verify

In [ ]:
# 2. Implementation Tasks & Forensic Engineering Questions
# 2.1 Required Code Implementation Tasks (Tasks 1–5)

In [ ]:

import os
import re
import sys
import json
import time
import hashlib
import subprocess
import platform
from types import SimpleNamespace

import numpy as np
import numba
from mpi4py import MPI
from numba import cuda, float64

# ------------------------------------------------------------------ constants
N = 512                          # global interior cells per direction
PY, PX = 2, 2                    # process grid
NY, NX = N // PY, N // PX        # 256 x 256 owned cells per rank
LAMBDA = 0.15
KAPPA_MIN, KAPPA_MAX = 0.08, 0.24
DX = DY = 1.0 / (N + 1)          # padded index 0 -> coordinate 0, N+1 -> coordinate 1
DT = 0.8 * DX * DX / (4.0 * KAPPA_MAX)      # explicit-scheme stability limit with margin
NSTEPS = 400
REPORT_EVERY = 100
TILE = 16                        # 16 x 16 thread block, 18 x 18 shared tile
TPB1D = 128                      # threads per block for pack/unpack kernels
MAX_ABS_DIFF_TOL = 1e-12
NULL = MPI.PROC_NULL
GITHUB_PLACEHOLDER = "zhansayamm"
GITHUB_USERNAME = os.environ.get("GITHUB_USERNAME", GITHUB_PLACEHOLDER).strip()


# =====================================================================
# Task 1 -- distributed topology and hardware-bound seed
# =====================================================================
def _gpu_uuid():
    try:
        return str(cuda.get_current_device().uuid)
    except Exception:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
            capture_output=True, text=True, check=True).stdout.strip().splitlines()[0]
        return out.strip()


def derive_instance_seed(github_username, gpu_uuid):
    """Deterministic 64-bit seed: first 8 bytes of SHA-256('<username>|<gpu uuid>')."""
    digest = hashlib.sha256(f"{github_username}|{gpu_uuid}".encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big")


def init_distributed_context():
    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    assert size == 4, f"This lab requires exactly 4 MPI ranks, got {size}"

    if GITHUB_USERNAME in ("", GITHUB_PLACEHOLDER):
        if rank == 0:
            print("ERROR: set your GitHub username first, e.g.  %env GITHUB_USERNAME=myname",
                  flush=True)
        sys.exit(1)

    # non-periodic 2 x 2 Cartesian communicator
    cart = comm.Create_cart(dims=[PY, PX], periods=[False, False], reorder=False)
    row_rank, col_rank = cart.Get_coords(cart.Get_rank())
    nbr_up, nbr_down = cart.Shift(0, 1)      # axis 0 (y): (source = row-1, dest = row+1)
    nbr_left, nbr_right = cart.Shift(1, 1)   # axis 1 (x): (source = col-1, dest = col+1)

    # all ranks share the single physical GPU
    cuda.select_device(0)
    gpu_uuid = _gpu_uuid()
    seed = derive_instance_seed(GITHUB_USERNAME, gpu_uuid)
    assert len(set(cart.allgather(seed))) == 1, "INSTANCE_SEED differs between ranks"

    return SimpleNamespace(
        comm=comm, cart=cart, rank=cart.Get_rank(), size=size,
        row_rank=row_rank, col_rank=col_rank,
        nbr={"up": nbr_up, "down": nbr_down, "left": nbr_left, "right": nbr_right},
        gpu_uuid=gpu_uuid, seed=seed,
        stream_boundary=cuda.stream(), stream_interior=cuda.stream(),
    )


# =====================================================================
# Task 2 -- seeded heterogeneous subdomain initialisation
# =====================================================================
def initialize_subdomains(ctx):
    """Builds kappa, S and the initial u for the local padded block (indices 0..257).
    Global coordinates come from the rank's Cartesian position, so no global
    512 x 512 array is ever allocated. All random parameters derive from INSTANCE_SEED."""
    rng = np.random.default_rng(ctx.seed)        # identical stream on every rank
    kx, ky = rng.integers(1, 4, size=2)
    phase = rng.uniform(0.0, 2.0 * np.pi, size=2)
    n_pulse = 3
    centers = rng.uniform(0.2, 0.8, size=(n_pulse, 2))     # (cx, cy)
    amps = rng.uniform(0.5, 1.5, size=n_pulse)
    widths = rng.uniform(0.05, 0.10, size=n_pulse)
    src_center = rng.uniform(0.3, 0.7, size=2)
    src_amp = float(rng.uniform(80.0, 120.0))
    src_sigma = 0.04

    # global physical coordinates of local indices 0..257
    gy = ctx.row_rank * NY + np.arange(NY + 2)
    gx = ctx.col_rank * NX + np.arange(NX + 2)
    Y, X = np.meshgrid(gy * DY, gx * DX, indexing="ij")

    # kappa in [0.08, 0.24]
    kappa = 0.16 + 0.08 * np.sin(2 * np.pi * kx * X + phase[0]) * np.cos(2 * np.pi * ky * Y + phase[1])
    kappa = np.clip(kappa, KAPPA_MIN, KAPPA_MAX)

    # initial Gaussian thermal pulses
    u0 = np.zeros_like(X)
    for (cx, cy), a, w in zip(centers, amps, widths):
        u0 += a * np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2.0 * w * w))

    # localized source, compact support (4 sigma)
    r2 = (X - src_center[0]) ** 2 + (Y - src_center[1]) ** 2
    S = src_amp * np.exp(-r2 / (2.0 * src_sigma ** 2))
    S[r2 > (4.0 * src_sigma) ** 2] = 0.0

    # homogeneous Dirichlet u = 0 on outer global edges (neighbour == PROC_NULL)
    if ctx.nbr["up"] == NULL:
        u0[0, :] = 0.0
    if ctx.nbr["down"] == NULL:
        u0[NY + 1, :] = 0.0
    if ctx.nbr["left"] == NULL:
        u0[:, 0] = 0.0
    if ctx.nbr["right"] == NULL:
        u0[:, NX + 1] = 0.0

    ctx.u0 = np.ascontiguousarray(u0, dtype=np.float64)
    ctx.d_kappa = cuda.to_device(np.ascontiguousarray(kappa, dtype=np.float64))
    ctx.d_S = cuda.to_device(np.ascontiguousarray(S, dtype=np.float64))

    ctx.kappa_min = ctx.cart.allreduce(float(kappa.min()), op=MPI.MIN)
    ctx.kappa_max = ctx.cart.allreduce(float(kappa.max()), op=MPI.MAX)
    ctx.init_params = {
        "kappa_wavenumbers": [int(kx), int(ky)],
        "kappa_phases": [float(p) for p in phase],
        "pulse_centers_xy": centers.tolist(),
        "pulse_amplitudes": amps.tolist(),
        "pulse_sigmas": widths.tolist(),
        "source_center_xy": src_center.tolist(),
        "source_amplitude": src_amp,
        "source_sigma": src_sigma,
    }

    # communication buffers: device vectors + page-locked host staging vectors
    sizes = {"up": NX, "down": NX, "left": NY, "right": NY}
    ctx.d_send = {k: cuda.device_array(n, dtype=np.float64) for k, n in sizes.items()}
    ctx.d_recv = {k: cuda.device_array(n, dtype=np.float64) for k, n in sizes.items()}
    ctx.h_send = {k: cuda.pinned_array(n, dtype=np.float64) for k, n in sizes.items()}
    ctx.h_recv = {k: cuda.pinned_array(n, dtype=np.float64) for k, n in sizes.items()}
    for k in sizes:
        ctx.h_send[k][:] = 0.0
        ctx.h_recv[k][:] = 0.0
    ctx.g_halo = (max(NX, NY) + TPB1D - 1) // TPB1D


# =====================================================================
# Task 3 -- GPU halo pack / unpack kernels and pinned asynchronous exchange
# =====================================================================
@cuda.jit
def pack_halos_kernel(u, send_up, send_down, send_left, send_right, ny, nx):
    """Owned border cells -> contiguous 1D float64 device send vectors."""
    t = cuda.grid(1)
    if t < nx:
        send_up[t] = u[1, t + 1]          # row 1
        send_down[t] = u[ny, t + 1]       # row ny
    if t < ny:
        send_left[t] = u[t + 1, 1]        # column 1
        send_right[t] = u[t + 1, nx]      # column nx


@cuda.jit
def unpack_halos_kernel(u, recv_up, recv_down, recv_left, recv_right, ny, nx):
    """Received vectors -> ghost rows / columns 0 and n+1."""
    t = cuda.grid(1)
    if t < nx:
        u[0, t + 1] = recv_up[t]
        u[ny + 1, t + 1] = recv_down[t]
    if t < ny:
        u[t + 1, 0] = recv_left[t]
        u[t + 1, nx + 1] = recv_right[t]


# tag = direction the data travels
TAG_UP, TAG_DOWN, TAG_LEFT, TAG_RIGHT = 0, 1, 2, 3


def exchange_halos_async(ctx, d_u, stream):
    """Stage A + posting of Stage B.
    Pack borders on `stream`, async D2H into pinned buffers, synchronise the stream so the
    host send buffers are valid, then post non-blocking Irecv/Isend with all four neighbours.
    Returns the list of MPI requests; finish with complete_halo_exchange()."""
    d_s, h_s, nb, cart = ctx.d_send, ctx.h_send, ctx.nbr, ctx.cart
    pack_halos_kernel[ctx.g_halo, TPB1D, stream](
        d_u, d_s["up"], d_s["down"], d_s["left"], d_s["right"], NY, NX)
    for k in ("up", "down", "left", "right"):
        if nb[k] != NULL:
            d_s[k].copy_to_host(h_s[k], stream=stream)
    stream.synchronize()

    h_r = ctx.h_recv
    return [
        # data I receive from the neighbour below travelled "up", and so on
        cart.Irecv(h_r["down"], source=nb["down"], tag=TAG_UP),
        cart.Irecv(h_r["up"], source=nb["up"], tag=TAG_DOWN),
        cart.Irecv(h_r["right"], source=nb["right"], tag=TAG_LEFT),
        cart.Irecv(h_r["left"], source=nb["left"], tag=TAG_RIGHT),
        cart.Isend(h_s["up"], dest=nb["up"], tag=TAG_UP),
        cart.Isend(h_s["down"], dest=nb["down"], tag=TAG_DOWN),
        cart.Isend(h_s["left"], dest=nb["left"], tag=TAG_LEFT),
        cart.Isend(h_s["right"], dest=nb["right"], tag=TAG_RIGHT),
    ]


def complete_halo_exchange(ctx, d_u, reqs, stream):
    """Stage C (first half): Waitall, async H2D on `stream`, unpack into ghost cells."""
    MPI.Request.Waitall(reqs)
    for k in ("up", "down", "left", "right"):
        if ctx.nbr[k] != NULL:
            ctx.d_recv[k].copy_to_device(ctx.h_recv[k], stream=stream)
    d_r = ctx.d_recv
    unpack_halos_kernel[ctx.g_halo, TPB1D, stream](
        d_u, d_r["up"], d_r["down"], d_r["left"], d_r["right"], NY, NX)


# =====================================================================
# Task 4 -- 2D shared-memory tiled stencil and dual-stream orchestrator
# =====================================================================
@cuda.jit
def tiled_stencil_kernel(u, un, kappa, S, r_start, r_end, c_start, c_end,
                         dt, inv_dx2, inv_dy2, lam):
    """Explicit update of cells [r_start, r_end) x [c_start, c_end) (half-open).
    16 x 16 thread block, 18 x 18 shared tile. Only cells that are really needed are
    loaded, so the deep-interior box never touches ghost cells."""
    smem = cuda.shared.array((TILE + 2, TILE + 2), float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    j = c_start + cuda.blockIdx.x * TILE + tx
    i = r_start + cuda.blockIdx.y * TILE + ty
    inside = (i < r_end) and (j < c_end)

    if inside:
        smem[ty + 1, tx + 1] = u[i, j]                       # centre cell
        if ty == 0:
            smem[0, tx + 1] = u[i - 1, j]                    # north apron
        if ty == TILE - 1 or i == r_end - 1:
            smem[ty + 2, tx + 1] = u[i + 1, j]               # south apron
        if tx == 0:
            smem[ty + 1, 0] = u[i, j - 1]                    # west apron
        if tx == TILE - 1 or j == c_end - 1:
            smem[ty + 1, tx + 2] = u[i, j + 1]               # east apron
    cuda.syncthreads()                                       # before any stencil arithmetic

    if inside:
        c = smem[ty + 1, tx + 1]
        uN = smem[ty, tx + 1]
        uS = smem[ty + 2, tx + 1]
        uW = smem[ty + 1, tx]
        uE = smem[ty + 1, tx + 2]
        k = kappa[i, j]
        kN = 0.5 * (kappa[i - 1, j] + k)
        kS = 0.5 * (kappa[i + 1, j] + k)
        kW = 0.5 * (kappa[i, j - 1] + k)
        kE = 0.5 * (kappa[i, j + 1] + k)
        diff_y = (kS * (uS - c) - kN * (c - uN)) * inv_dy2
        diff_x = (kE * (uE - c) - kW * (c - uW)) * inv_dx2
        un[i, j] = c + dt * (diff_x + diff_y - lam * c * c * c + S[i, j])


def launch_stencil(ctx, stream, d_u, d_un, r_start, r_end, c_start, c_end):
    if r_end <= r_start or c_end <= c_start:
        return
    grid = ((c_end - c_start + TILE - 1) // TILE, (r_end - r_start + TILE - 1) // TILE)
    tiled_stencil_kernel[grid, (TILE, TILE), stream](
        d_u, d_un, ctx.d_kappa, ctx.d_S, r_start, r_end, c_start, c_end,
        DT, 1.0 / (DX * DX), 1.0 / (DY * DY), LAMBDA)


def step_serialized(ctx, d_u, d_un):
    """Baseline: complete halo exchange first, then one full-domain stencil. No overlap."""
    sb = ctx.stream_boundary
    reqs = exchange_halos_async(ctx, d_u, sb)
    complete_halo_exchange(ctx, d_u, reqs, sb)
    launch_stencil(ctx, sb, d_u, d_un, 1, NY + 1, 1, NX + 1)
    sb.synchronize()


def step_overlapped(ctx, d_u, d_un):
    """Dual-stream pipeline: deep interior on stream_interior overlaps the MPI exchange,
    the outer 1-cell ring runs on stream_boundary once the ghosts are in place."""
    sb, si = ctx.stream_boundary, ctx.stream_interior
    # Stage A + B: pack, D2H, sync, post Irecv/Isend ...
    reqs = exchange_halos_async(ctx, d_u, sb)
    # ... and immediately launch the deep interior [2..ny-1] x [2..nx-1]
    launch_stencil(ctx, si, d_u, d_un, 2, NY, 2, NX)
    # Stage C: Waitall, H2D, unpack, boundary ring
    complete_halo_exchange(ctx, d_u, reqs, sb)
    launch_stencil(ctx, sb, d_u, d_un, 1, 2, 1, NX + 1)         # row 1
    launch_stencil(ctx, sb, d_u, d_un, NY, NY + 1, 1, NX + 1)   # row ny
    launch_stencil(ctx, sb, d_u, d_un, 2, NY, 1, 2)             # column 1
    launch_stencil(ctx, sb, d_u, d_un, 2, NY, NX, NX + 1)       # column nx
    sb.synchronize()
    si.synchronize()


# =====================================================================
# Task 5 -- reductions, benchmark, PTX audit, manifest
# =====================================================================
def global_diagnostics(ctx, step, d_old, d_new):
    """E = sum(u * dx * dy) and L2 residual ||u^{n+1} - u^n||_2, via Allreduce(SUM)."""
    new = d_new.copy_to_host()[1:NY + 1, 1:NX + 1]
    old = d_old.copy_to_host()[1:NY + 1, 1:NX + 1]
    local = np.array([np.sum(new) * DX * DY, np.sum((new - old) ** 2)], dtype=np.float64)
    glob = np.empty(2, dtype=np.float64)
    ctx.cart.Allreduce(local, glob, op=MPI.SUM)
    return {"step": step, "energy": float(glob[0]), "l2_residual": float(np.sqrt(glob[1]))}


def run_mode(ctx, name, step_fn, nsteps=NSTEPS, report_every=REPORT_EVERY):
    d_u = cuda.to_device(ctx.u0)
    d_un = cuda.to_device(ctx.u0)
    cuda.synchronize()
    ctx.cart.Barrier()
    t_diag = 0.0
    history = []
    t0 = MPI.Wtime()
    for n in range(1, nsteps + 1):
        step_fn(ctx, d_u, d_un)
        if n % report_every == 0:
            td = MPI.Wtime()
            history.append(global_diagnostics(ctx, n, d_u, d_un))   # d_u = old, d_un = new
            t_diag += MPI.Wtime() - td
        d_u, d_un = d_un, d_u
    cuda.synchronize()
    ctx.cart.Barrier()
    elapsed = (MPI.Wtime() - t0) - t_diag
    final = d_u.copy_to_host()[1:NY + 1, 1:NX + 1].copy()
    if ctx.rank == 0:
        print(f"[{name:>10}] {nsteps} steps in {elapsed:.3f} s  ({elapsed / nsteps * 1e3:.3f} ms/step)")
        for h in history:
            print(f"             step {h['step']:4d}  E = {h['energy']:.12e}  "
                  f"||du||_2 = {h['l2_residual']:.6e}")
    return {"time_s": elapsed, "history": history, "final": final}


def ptx_audit(path="stencil_kernel.ptx"):
    """Static audit of the tiled stencil PTX: shared-memory use, barrier placement, no spills."""
    try:
        asm = tiled_stencil_kernel.inspect_asm()
        if isinstance(asm, dict):
            asm = next(iter(asm.values()))
            if isinstance(asm, dict):
                asm = next(iter(asm.values()))
        ptx = str(asm)
    except Exception as exc:                                   # noqa: BLE001
        return {"available": False, "error": repr(exc), "passed": False}

    with open(path, "w") as f:
        f.write(ptx)

    def pos(pattern):
        return [m.start() for m in re.finditer(pattern, ptx)]

    bar = pos(r"\bbar\.sync\b")
    ld_sh = pos(r"\bld\.shared")
    st_sh = pos(r"\bst\.shared")
    ld_gl = pos(r"\bld\.global")
    st_gl = pos(r"\bst\.global")
    local_ops = pos(r"\b(?:ld|st)\.local")
    fma64 = pos(r"\bfma\.rn\.f64")
    shared_decl = re.findall(r"\.shared[^;\n]*\[(\d+)\]", ptx)
    shared_bytes = sum(int(x) for x in shared_decl)
    target = re.search(r"\.target\s+(\S+)", ptx)

    ordered = bool(bar and st_sh and ld_sh and max(st_sh) < min(bar) < min(ld_sh))
    expected_smem = (TILE + 2) ** 2 * 8
    checks = {
        "has_bar_sync": len(bar) >= 1,
        "shared_stores_before_barrier_and_loads_after": ordered,
        "shared_tile_bytes_match_18x18_f64": shared_bytes == expected_smem,
        "no_local_memory_spills": len(local_ops) == 0,
    }
    return {
        "available": True,
        "target": target.group(1) if target else None,
        "ptx_sha256": hashlib.sha256(ptx.encode()).hexdigest(),
        "ptx_file": path,
        "counts": {
            "bar.sync": len(bar), "ld.shared": len(ld_sh), "st.shared": len(st_sh),
            "ld.global": len(ld_gl), "st.global": len(st_gl),
            "ld/st.local": len(local_ops), "fma.rn.f64": len(fma64),
        },
        "shared_bytes": shared_bytes,
        "expected_shared_bytes": expected_smem,
        "checks": checks,
        "passed": all(checks.values()),
    }


def run_solver(ctx):
    cart = ctx.cart

    # JIT warm-up of every kernel (kept out of the timed region)
    wu, wn = cuda.to_device(ctx.u0), cuda.to_device(ctx.u0)
    step_serialized(ctx, wu, wn)
    step_overlapped(ctx, wu, wn)
    cuda.synchronize()
    cart.Barrier()

    ser = run_mode(ctx, "serialized", step_serialized)
    ovl = run_mode(ctx, "overlapped", step_overlapped)

    # serialized vs overlapped must agree to machine precision
    local_diff = float(np.max(np.abs(ser["final"] - ovl["final"])))
    max_diff = cart.allreduce(local_diff, op=MPI.MAX)
    diff_ok = max_diff < MAX_ABS_DIFF_TOL

    ck_ser = cart.gather(hashlib.sha256(ser["final"].tobytes()).hexdigest(), root=0)
    ck_ovl = cart.gather(hashlib.sha256(ovl["final"].tobytes()).hexdigest(), root=0)

    audit = ptx_audit() if ctx.rank == 0 else None
    kappa_ok = (ctx.kappa_min >= KAPPA_MIN - 1e-15) and (ctx.kappa_max <= KAPPA_MAX + 1e-15)

    if ctx.rank == 0:
        speedup = ser["time_s"] / ovl["time_s"]
        print(f"max |serialized - overlapped| = {max_diff:.3e}   "
              f"({'PASS' if diff_ok else 'FAIL'}, tol {MAX_ABS_DIFF_TOL:g})")
        print(f"speedup (serialized / overlapped) = {speedup:.3f}x")
        print(f"kappa range = [{ctx.kappa_min:.4f}, {ctx.kappa_max:.4f}]")
        print("PTX audit:", "PASS" if audit["passed"] else "FAIL")
        if audit.get("available"):
            print("  counts:", audit["counts"], " shared bytes:", audit["shared_bytes"])
            for k, v in audit["checks"].items():
                print(f"  - {k}: {v}")
        else:
            print("  unavailable:", audit.get("error"))

        dev = cuda.get_current_device()
        name = dev.name.decode() if isinstance(dev.name, bytes) else str(dev.name)
        manifest = {
            "github_username": GITHUB_USERNAME,
            "gpu_uuid": ctx.gpu_uuid,
            "gpu_name": name,
            "compute_capability": list(dev.compute_capability),
            "instance_seed_uint64": ctx.seed,
            "instance_seed_hex": f"0x{ctx.seed:016x}",
            "seed_derivation": "first 8 bytes (big-endian) of SHA-256('<github_username>|<gpu_uuid>')",
            "problem": {"N": N, "ranks": ctx.size, "process_grid": [PY, PX],
                        "subdomain": [NY, NX], "padded": [NY + 2, NX + 2],
                        "dx": DX, "dt": DT, "lambda": LAMBDA,
                        "kappa_range_spec": [KAPPA_MIN, KAPPA_MAX],
                        "kappa_range_observed": [ctx.kappa_min, ctx.kappa_max],
                        "nsteps": NSTEPS, "report_every": REPORT_EVERY},
            "init_params": ctx.init_params,
            "serialized": {"time_s": ser["time_s"], "history": ser["history"]},
            "overlapped": {"time_s": ovl["time_s"], "history": ovl["history"]},
            "speedup_serialized_over_overlapped": speedup,
            "max_abs_diff_serialized_vs_overlapped": max_diff,
            "max_abs_diff_tolerance": MAX_ABS_DIFF_TOL,
            "final_state_match": bool(diff_ok),
            "bitwise_identical": ck_ser == ck_ovl,
            "final_state_sha256_per_rank": {"serialized": ck_ser, "overlapped": ck_ovl},
            "ptx_audit": audit,
            "environment": {
                "python": platform.python_version(),
                "numba": numba.__version__,
                "numpy": np.__version__,
                "mpi_library": MPI.Get_library_version().strip().splitlines()[0],
            },
            "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "all_checks_passed": bool(diff_ok and kappa_ok and audit["passed"]),
        }
        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        print("wrote submission_manifest.json")

    assert diff_ok, f"serialized vs overlapped mismatch: {max_diff:.3e}"


def main():
    ctx = init_distributed_context()
    if ctx.rank == 0:
        print(f"INSTANCE_SEED = 0x{ctx.seed:016x}   GPU UUID = {ctx.gpu_uuid}")
    initialize_subdomains(ctx)
    run_solver(ctx)


if __name__ == "__main__":
    main()

In [11]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python lab_solver.py